<a href="https://colab.research.google.com/github/NMapelu/AirlineChatbot/blob/main/notebooks/02_preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
import numpy as np
import re
import os
import json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

DRIVE_ROOT = "/content/drive/MyDrive/AirlineChatbot"
RAW_DATA = f"{DRIVE_ROOT}/data/raw/bitext_travel.csv"
PROCESSED_DIR = f"{DRIVE_ROOT}/data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Raw data path :", RAW_DATA)
print("Processed dir :", PROCESSED_DIR)
print("Raw file exists:", os.path.exists(RAW_DATA))

Raw data path : /content/drive/MyDrive/AirlineChatbot/data/raw/bitext_travel.csv
Processed dir : /content/drive/MyDrive/AirlineChatbot/data/processed
Raw file exists: True


In [5]:
df = pd.read_csv(RAW_DATA)

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nNull values per column:")
print(df.isnull().sum())

Shape: (31658, 5)

Columns: ['instruction', 'intent', 'category', 'tags', 'response']

First 5 rows:
                                         instruction                   intent  \
0  I want to know about my checked carry-on bagga...  check_baggage_allowance   
1  I'm looking for information about my checked c...  check_baggage_allowance   
2  I'm looking for information about my fucking c...  check_baggage_allowance   
3  I have to see the fuking checked carry-on bagg...  check_baggage_allowance   
4  need to see the fucking checkedcarry-on baggag...  check_baggage_allowance   

  category     tags                                           response  
0  BAGGAGE     BCIL  To find out your checked baggage allowance, pl...  
1  BAGGAGE    BCLQZ  To check the details of your checked baggage a...  
2  BAGGAGE    BCILW  To find out your checked baggage allowance, pl...  
3  BAGGAGE  BCILPWZ  To review your checked baggage allowance, plea...  
4  BAGGAGE  BCILQWZ  To check your baggage allo

In [6]:
TEXT_COL = "instruction"
LABEL_COL = "intent"

assert TEXT_COL in df.columns, f"'{TEXT_COL}' not found. Available: {df.columns.tolist()}"
assert LABEL_COL in df.columns, f"'{LABEL_COL}' not found. Available: {df.columns.tolist()}"

print(f"Using text column : '{TEXT_COL}'")
print(f"Using label column: '{LABEL_COL}'")

Using text column : 'instruction'
Using label column: 'intent'


In [7]:
print("Number of unique intents:", df[LABEL_COL].nunique())
print("\nIntent distribution:")
print(df[LABEL_COL].value_counts())
print("\nMin class count:", df[LABEL_COL].value_counts().min())
print("Max class count:", df[LABEL_COL].value_counts().max())

Number of unique intents: 33

Intent distribution:
intent
check_trip_prices                  1874
check_flight_insurance_coverage     993
cancel_flight                       992
check_cancellation_fee              992
check_trip_details                  984
check_flight_reservation            983
change_trip                         983
check_trip_insurance_coverage       978
search_flight_insurance             978
print_boarding_pass                 975
change_flight                       975
check_trip_plan                     974
change_seat                         971
check_baggage_allowance             970
check_trip_offers                   963
book_flight                         963
purchase_trip_insurance             960
check_flight_prices                 956
search_trip                         952
check_flight_offers                 947
check_flight_status                 947
search_flight                       946
cancel_trip                         940
purchase_flight_insura

In [8]:
def clean_text(text):
    """Basic normalization: lowercase, collapse whitespace, strip."""
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)   # collapse multiple spaces/newlines
    return text

df["text"] = df[TEXT_COL].apply(clean_text)

# Preview
print("Sample cleaned texts:")
for t in df["text"].head(5):
    print(" •", t)

Sample cleaned texts:
 • i want to know about my checked carry-on baggage allowance, how can i get more information?
 • i'm looking for information about my checked carry-on baggage allowance. help me.
 • i'm looking for information about my fucking checked carry-on baggage allowance, where can i get it?
 • i have to see the fuking checked carry-on baggage allowance, could i get some help?
 • need to see the fucking checkedcarry-on baggage allowance how to do it


In [9]:
before = len(df)

# Drop rows with null text or label
df = df.dropna(subset=["text", LABEL_COL])

# Drop exact duplicate texts (keep first)
df = df.drop_duplicates(subset=["text"], keep="first")

after = len(df)

print(f"Rows before: {before}")
print(f"Rows after : {after}")
print(f"Removed    : {before - after}")

Rows before: 31658
Rows after : 30601
Removed    : 1057


In [10]:
le = LabelEncoder()
df["label"] = le.fit_transform(df[LABEL_COL])

# Build and save the label mapping
label_map = {int(v): str(k) for k, v in zip(le.classes_, le.transform(le.classes_))}

print(f"Number of classes: {len(label_map)}")
print("\nLabel mapping (id -> intent):")
for k, v in label_map.items():
    print(f"  {k:3d} -> {v}")

Number of classes: 33

Label mapping (id -> intent):
    0 -> book_flight
    1 -> book_trip
    2 -> cancel_flight
    3 -> cancel_trip
    4 -> change_flight
    5 -> change_seat
    6 -> change_trip
    7 -> check_arrival_time
    8 -> check_baggage_allowance
    9 -> check_cancellation_fee
   10 -> check_departure_time
   11 -> check_flight_insurance_coverage
   12 -> check_flight_offers
   13 -> check_flight_prices
   14 -> check_flight_reservation
   15 -> check_flight_status
   16 -> check_in
   17 -> check_trip_details
   18 -> check_trip_insurance_coverage
   19 -> check_trip_offers
   20 -> check_trip_plan
   21 -> check_trip_prices
   22 -> choose_seat
   23 -> get_boarding_pass
   24 -> get_refund
   25 -> human_agent
   26 -> print_boarding_pass
   27 -> purchase_flight_insurance
   28 -> purchase_trip_insurance
   29 -> search_flight
   30 -> search_flight_insurance
   31 -> search_trip
   32 -> search_trip_insurance


In [11]:
# 80% train, 10% val, 10% test — stratified to preserve class balance
train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42
)

print(f"Train size: {len(train_df)}  ({len(train_df)/len(df)*100:.1f}%)")
print(f"Val size  : {len(val_df)}  ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test size : {len(test_df)}  ({len(test_df)/len(df)*100:.1f}%)")

print("\nClasses in train:", train_df["label"].nunique())
print("Classes in val  :", val_df["label"].nunique())
print("Classes in test :", test_df["label"].nunique())

Train size: 24480  (80.0%)
Val size  : 3060  (10.0%)
Test size : 3061  (10.0%)

Classes in train: 33
Classes in val  : 33
Classes in test : 33


In [12]:
# Save splits
train_df[["text", "label", LABEL_COL]].to_csv(f"{PROCESSED_DIR}/train.csv", index=False)
val_df[["text", "label", LABEL_COL]].to_csv(f"{PROCESSED_DIR}/val.csv", index=False)
test_df[["text", "label", LABEL_COL]].to_csv(f"{PROCESSED_DIR}/test.csv", index=False)

# Save label mapping
with open(f"{PROCESSED_DIR}/label_map.json", "w") as f:
    json.dump(label_map, f, indent=2)

print("Saved files:")
for f in sorted(os.listdir(PROCESSED_DIR)):
    path = f"{PROCESSED_DIR}/{f}"
    size_kb = os.path.getsize(path) / 1024
    print(f"  {f}  ({size_kb:.1f} KB)")

Saved files:
  label_map.json  (0.9 KB)
  test.csv  (245.5 KB)
  train.csv  (1960.0 KB)
  val.csv  (244.4 KB)


In [13]:
# Reload from Drive to confirm everything is saved properly
train_check = pd.read_csv(f"{PROCESSED_DIR}/train.csv")
val_check = pd.read_csv(f"{PROCESSED_DIR}/val.csv")
test_check = pd.read_csv(f"{PROCESSED_DIR}/test.csv")

with open(f"{PROCESSED_DIR}/label_map.json") as f:
    label_map_check = json.load(f)

print("Train shape:", train_check.shape)
print("Val shape  :", val_check.shape)
print("Test shape :", test_check.shape)
print("Classes    :", len(label_map_check))

print("\nSample rows from train:")
print(train_check.head(3))

print("\nLabel distribution in train:")
print(train_check["label"].value_counts().sort_index())

Train shape: (24480, 3)
Val shape  : (3060, 3)
Test shape : (3061, 3)
Classes    : 33

Sample rows from train:
                                                text  label         intent
0  wanna modify my fucking trip to madrid could i...      6    change_trip
1  wanna know about excursions to madrid where co...     31    search_trip
2  i would like to change a flight booking, how d...      4  change_flight

Label distribution in train:
label
0      762
1      702
2      787
3      744
4      771
5      720
6      776
7      616
8      750
9      774
10     615
11     754
12     752
13     750
14     777
15     738
16     644
17     783
18     750
19     766
20     762
21    1445
22     544
23     671
24     693
25     630
26     599
27     685
28     757
29     751
30     746
31     752
32     714
Name: count, dtype: int64


In [14]:
print("=" * 60)
print("SANITY CHECK: Random samples per intent")
print("=" * 60)

for label_id in sorted(train_check["label"].unique())[:5]:
    subset = train_check[train_check["label"] == label_id]
    intent_name = label_map_check[str(label_id)]
    sample = subset["text"].iloc[0]
    print(f"\n[{label_id}] {intent_name}")
    print(f"  → {sample}")

SANITY CHECK: Random samples per intent

[0] book_flight
  → i need to reserve a fucking flight from chicago to madrid, where can i do it?

[1] book_trip
  → i have to buy a excursion to madrid, could i get some help?

[2] cancel_flight
  → i cannot travel, help me cancel my fucking flight booking from chicago to madrid

[3] cancel_trip
  → i cannot travel help me tocancel my fucking getaway

[4] change_flight
  → i would like to change a flight booking, how do i do it?


In [16]:
print("=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)
print(f"Raw data        : {RAW_DATA}")
print(f"Processed files : {PROCESSED_DIR}")
print(f"Total examples  : {len(df)}")
print(f"Classes         : {len(label_map)}")
print(f"Train/Val/Test  : {len(train_df)}/{len(val_df)}/{len(test_df)}")


PREPROCESSING COMPLETE
Raw data        : /content/drive/MyDrive/AirlineChatbot/data/raw/bitext_travel.csv
Processed files : /content/drive/MyDrive/AirlineChatbot/data/processed
Total examples  : 30601
Classes         : 33
Train/Val/Test  : 24480/3060/3061
